# MSM Notebook 2 — INGESTION LAYER (raw only)

**확정 셋 (이전 세션):**
- Q1=A: KRX 전체 + 6축 필터 (1000억 / 3억 / 120일 / FLAG / KRX / ETF 제외)
- Q2=A: OHLCV(adj+raw) + Flow + Foreign
- Q3=A: KOSPI200, KOSDAQ150, USD/KRW, 한국 기준금리 (ECOS)
- Q6=B: raw 수집만. normalization은 Notebook 3.

**MSM 단독 시스템.** premium repo 참조 X.

**기간:** 2017-01-01 ~ 2026-06-02 (10년)

**산출 위치:**
- `data/raw/active/ohlcv_adj/year=YYYY.parquet`
- `data/raw/active/ohlcv/year=YYYY.parquet`
- `data/raw/active/flow/year=YYYY.parquet`
- `data/raw/active/foreign/year=YYYY.parquet`
- `data/raw/macro/macro_10y.json`
- `data/universe/universe_msm.csv`

---
## 0. Setup

In [ ]:
# Colab 환경 setup
!pip install -q pykrx pyarrow finance-datareader requests

In [ ]:
import os, json, time
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
import requests

# 인증 (KRX + ECOS)
from google.colab import userdata
os.environ["KRX_ID"] = userdata.get("KRX_ID")
os.environ["KRX_PW"] = userdata.get("KRX_PW")
ECOS_KEY = userdata.get("ECOS_API_KEY")
GH_TOKEN = userdata.get("GH_TOKEN")
print(f"KRX_ID: {bool(os.environ.get('KRX_ID'))}, ECOS: {bool(ECOS_KEY)}, GH: {bool(GH_TOKEN)}")

# repo 이동
%cd /content/choonsimi-msm
!pwd

from pykrx import stock as krx
import FinanceDataReader as fdr

# 기간
START = "20170101"
END   = datetime.now().strftime("%Y%m%d")
print(f"\n수집 기간: {START} ~ {END}")

# 디렉토리
ROOT = Path("data")
(ROOT / "raw" / "active" / "ohlcv_adj").mkdir(parents=True, exist_ok=True)
(ROOT / "raw" / "active" / "ohlcv").mkdir(parents=True, exist_ok=True)
(ROOT / "raw" / "active" / "flow").mkdir(parents=True, exist_ok=True)
(ROOT / "raw" / "active" / "foreign").mkdir(parents=True, exist_ok=True)
(ROOT / "raw" / "macro").mkdir(parents=True, exist_ok=True)
(ROOT / "universe").mkdir(parents=True, exist_ok=True)
(ROOT / "checks").mkdir(parents=True, exist_ok=True)

REPORT = {"start": START, "end": END, "steps": {}}
print("디렉토리 준비 완료")

---
## 검증 셀 — pykrx 지수 코드 확인

**§1 추측 금지** — 매뉴얼에 명시 안 된 지수 코드는 호출로 검증.

확정값 (매뉴얼 §6):
- KOSPI200: `1028` ✓
- VKOSPI: `1330` + name_display=False ✓ (이미 보유)

확인 대상:
- KOSPI 종합 (1001?)
- KOSDAQ 종합 (2001?)
- KOSDAQ150 (2203?)

In [ ]:
# pykrx 지수 전체 list에서 정확 코드 찾기
kospi_idx = krx.get_index_ticker_list(END, market="KOSPI")
kosdaq_idx = krx.get_index_ticker_list(END, market="KOSDAQ")
print(f"KOSPI 지수 수: {len(kospi_idx)}, KOSDAQ 지수 수: {len(kosdaq_idx)}")

# 이름으로 매핑
target_names = ["코스피", "코스닥", "코스피 200", "코스닥 150", "KRX 100"]
found = {}
for t in kospi_idx + kosdaq_idx:
    try:
        name = krx.get_index_ticker_name(t)
        for tn in target_names:
            if tn in name:
                found.setdefault(tn, []).append((t, name))
    except Exception:
        pass

print("\n=== 발견 지수 ===")
for k, v in found.items():
    print(f"\n{k}:")
    for t, n in v:
        print(f"  {t}: {n}")

# REPORT에 기록 (다음 셀에서 사용)
REPORT["steps"]["index_codes"] = {k: [t for t, _ in v] for k, v in found.items()}

---
## Step A — Universe 선정 (6축 필터)

**컷오프:**
- 시총 ≥ 1,000억 KRW
- 60일 평균 거래대금 ≥ 3억 KRW
- 상장 ≥ 120 trading days
- corporate action: FLAG만 (제외 X)
- 시장: KOSPI + KOSDAQ
- ETF/ETN: 제외

**시점:** 오늘 (END) 기준 1회 선정. 분기 재선정은 별도.

In [ ]:
# A-1. 전체 ticker list (오늘 기준)
kospi_codes  = krx.get_market_ticker_list(END, market="KOSPI")
kosdaq_codes = krx.get_market_ticker_list(END, market="KOSDAQ")
all_codes = sorted(set(kospi_codes) | set(kosdaq_codes))
print(f"KRX 전체: KOSPI {len(kospi_codes)} + KOSDAQ {len(kosdaq_codes)} = {len(all_codes)}")

In [ ]:
# A-2. ETF/ETN 제외용 — pykrx ETF/ETN list 받기
etf_codes = set(krx.get_etf_ticker_list(END))
etn_codes = set(krx.get_etn_ticker_list(END))
print(f"ETF: {len(etf_codes)}, ETN: {len(etn_codes)}")

filtered_codes = [c for c in all_codes if c not in etf_codes and c not in etn_codes]
print(f"ETF/ETN 제외 후: {len(filtered_codes)}")

In [ ]:
# A-3. 시총 + 60일 거래대금 (오늘 기준 batch)
df_cap = krx.get_market_cap_by_ticker(END, market="ALL")
df_cap = df_cap.reset_index().rename(columns={
    "티커": "code", "종가": "close", "시가총액": "market_cap",
    "거래량": "volume", "거래대금": "trade_value", "상장주식수": "shares"
})
df_cap["code"] = df_cap["code"].astype(str).str.zfill(6)
print(f"시총 데이터: {len(df_cap)}개")
print(df_cap.head(3))

In [ ]:
# A-4. 60일 평균 거래대금 — 전체 종목 batch (영업일 60일치 누적)
from datetime import timedelta
end_dt = datetime.strptime(END, "%Y%m%d")
start_60d = (end_dt - timedelta(days=120)).strftime("%Y%m%d")  # 영업일 60≈달력 90

# 일자별 batch (1콜 = 전종목 1일)
trdval_60d = {}
from pykrx.stock import get_market_ohlcv_by_ticker

# 실제 영업일 받기
biz_days = krx.get_previous_business_days(year=end_dt.year, month=end_dt.month)
print(f"이번 달 영업일: {len(biz_days)}일")

# 60일치 영업일 (단순: 최근 60 영업일)
# get_market_ohlcv_by_ticker 일별 호출 → 너무 느림
# 대안: 종목별 호출 (단 cap 종목만)
print("\n주의: A-4는 시간 오래 걸림. A-5에서 종목별로 직접 호출 (lookback 120 영업일)")

In [ ]:
# A-5. 시총 1000억 1차 필터 → 그 종목만 60일 거래대금 + 상장기간 계산
MIN_CAP = 1_000 * 1e8   # 1000억
MIN_TRDVAL_60D = 3 * 1e8  # 3억
MIN_LISTED_DAYS = 120

# 1차: 시총
cap_pass = df_cap[df_cap["market_cap"] >= MIN_CAP].copy()
cap_pass = cap_pass[cap_pass["code"].isin(filtered_codes)]  # ETF/ETN 제외 적용
print(f"시총 ≥ 1000억 통과: {len(cap_pass)}종목")

In [ ]:
# A-6. 시총 통과 종목 — 60일 거래대금 + 상장기간 계산
import time
lookback_start = (end_dt - timedelta(days=200)).strftime("%Y%m%d")  # 영업일 120 + 여유

rows = []
fail = []
for i, code in enumerate(cap_pass["code"].tolist(), 1):
    try:
        df = krx.get_market_ohlcv(lookback_start, END, code, adjusted=False)
        if len(df) < MIN_LISTED_DAYS:
            continue  # 상장기간 부족
        # 거래대금: 매뉴얼 §4.2 — adjusted=False 일 때 "거래대금" 컬럼 존재
        if "거래대금" in df.columns:
            trdval_60d = df["거래대금"].tail(60).mean()
        else:
            # raw close × volume 근사
            trdval_60d = (df["종가"] * df["거래량"]).tail(60).mean()
        if trdval_60d < MIN_TRDVAL_60D:
            continue
        rows.append({
            "code": code,
            "avg_trdval_60d": int(trdval_60d),
            "listed_days": len(df),
        })
    except Exception as e:
        fail.append((code, str(e)[:40]))
    time.sleep(0.15)
    if i % 50 == 0:
        print(f"  진행: {i}/{len(cap_pass)} (통과: {len(rows)})")

df_pass = pd.DataFrame(rows)
print(f"\n2~3차 필터 통과: {len(df_pass)}종목 (실패 {len(fail)})")

In [ ]:
# A-7. Universe 최종 — 종목명·시장 합치고 저장
df_univ = df_pass.merge(df_cap[["code", "market_cap", "shares"]], on="code")
df_univ["name"] = df_univ["code"].apply(lambda c: krx.get_market_ticker_name(c))
df_univ["market"] = df_univ["code"].apply(
    lambda c: "KOSPI" if c in kospi_codes else "KOSDAQ"
)
df_univ = df_univ[["code", "name", "market", "market_cap", "shares", "avg_trdval_60d", "listed_days"]]
df_univ = df_univ.sort_values("market_cap", ascending=False).reset_index(drop=True)

univ_path = ROOT / "universe" / "universe_msm.csv"
df_univ.to_csv(univ_path, index=False, encoding="utf-8-sig")
print(f"Universe 저장: {univ_path}")
print(f"  종목수: {len(df_univ)}")
print(f"  KOSPI: {(df_univ['market']=='KOSPI').sum()}, KOSDAQ: {(df_univ['market']=='KOSDAQ').sum()}")
print(df_univ.head(10))

REPORT["steps"]["universe"] = {
    "count": len(df_univ),
    "kospi": int((df_univ['market']=='KOSPI').sum()),
    "kosdaq": int((df_univ['market']=='KOSDAQ').sum()),
    "min_cap": int(df_univ['market_cap'].min()),
    "path": str(univ_path),
}

---
## Step B — Active OHLCV adjusted (배치 수집)

**매뉴얼 §4.1:** `get_market_ohlcv(start, end, code, adjusted=True)` 종목별 1콜.
**예상 시간:** universe 종목수 × 2.4초 (10년).

**Batch 단위:** 50종목씩. 매 batch 종료 시 push (Notebook 1 §13).

In [ ]:
# B-1. Universe 로드
df_univ = pd.read_csv(ROOT / "universe" / "universe_msm.csv", encoding="utf-8-sig", dtype={"code": str})
df_univ["code"] = df_univ["code"].str.zfill(6)
codes = df_univ["code"].tolist()
print(f"수집 대상: {len(codes)}종목")

BATCH_SIZE = 50
n_batches = (len(codes) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"batch 수: {n_batches}")

# progress 파일
progress_path = ROOT / "checks" / "active_progress.json"
if not progress_path.exists():
    with open(progress_path, "w") as f:
        json.dump({"completed_batches": []}, f)

In [ ]:
def run_active_ohlcv_adj_batch(batch_num):
    """Active OHLCV adjusted batch 수집"""
    start = batch_num * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(codes))
    batch_codes = codes[start:end]
    print(f"=== Batch {batch_num}: codes[{start}:{end}] ({len(batch_codes)}개) ===")
    
    rows = []
    fail = []
    for i, code in enumerate(batch_codes, 1):
        try:
            df = krx.get_market_ohlcv(START, END, code, adjusted=True)
            if len(df) > 0:
                df = df.reset_index().rename(columns={
                    "날짜": "date", "시가": "open", "고가": "high", "저가": "low",
                    "종가": "close", "거래량": "volume", "등락률": "change_rate"
                })
                df["code"] = code.zfill(6)
                rows.append(df)
        except Exception as e:
            fail.append((code, str(e)[:40]))
        time.sleep(0.15)
        if i % 10 == 0:
            print(f"  진행: {i}/{len(batch_codes)}")
    
    # 저장 (연도별 incremental)
    if rows:
        df_all = pd.concat(rows, ignore_index=True)
        df_all["date"] = pd.to_datetime(df_all["date"])
        df_all["year"] = df_all["date"].dt.year
        for year, df_y in df_all.groupby("year"):
            out = ROOT / "raw" / "active" / "ohlcv_adj" / f"year={year}.parquet"
            df_y_new = df_y.drop(columns=["year"])
            if out.exists():
                df_old = pd.read_parquet(out)
                df_y_new = pd.concat([df_old, df_y_new], ignore_index=True).drop_duplicates(
                    subset=["code", "date"], keep="last"
                )
            df_y_new.to_parquet(out, index=False)
        print(f"✓ Batch {batch_num} 저장: {len(df_all)}행, {df_all['code'].nunique()}종목, 실패 {len(fail)}")
    return {"batch": batch_num, "rows": len(df_all) if rows else 0, "fail": len(fail)}

print("함수 준비. 실행: run_active_ohlcv_adj_batch(0), (1), ... (n-1)")

---
## Step C — Active OHLCV raw + 시총 (일자별 batch)

**매뉴얼 §4.2:** `get_market_cap_by_ticker(date, market)` 일자별 1콜에 전종목.
**예상:** 10년 영업일 ~2500 × 0.9초 ≈ 40분.

**저장:** 연도별 파일. universe 종목만 필터링.

**주의:** 매뉴얼 §4.2 — raw 가격은 trade_value, market_cap 컬럼 보유.

In [ ]:
def collect_active_ohlcv_raw_by_year(year):
    """일자별 cap batch → 해당 연도만 누적 → universe 종목만 저장"""
    yr_start = f"{year}0101"
    yr_end   = f"{year}1231" if year < int(END[:4]) else END
    
    # 영업일 list
    biz_days_str = krx.get_previous_business_days(fromdate=yr_start, todate=yr_end)
    biz_days_str = [d.strftime("%Y%m%d") for d in biz_days_str]
    print(f"{year}년: {len(biz_days_str)} 영업일")
    
    univ_codes = set(codes)
    rows = []
    fail = []
    for i, d in enumerate(biz_days_str, 1):
        try:
            df = krx.get_market_cap_by_ticker(d, market="ALL")
            df = df.reset_index().rename(columns={
                "티커": "code", "종가": "close", "시가총액": "market_cap",
                "거래량": "volume", "거래대금": "trade_value", "상장주식수": "shares"
            })
            df["code"] = df["code"].astype(str).str.zfill(6)
            df = df[df["code"].isin(univ_codes)].copy()
            df["date"] = pd.to_datetime(d)
            rows.append(df)
        except Exception as e:
            fail.append((d, str(e)[:40]))
        time.sleep(0.1)
        if i % 50 == 0:
            print(f"  진행: {i}/{len(biz_days_str)}")
    
    if rows:
        df_all = pd.concat(rows, ignore_index=True)
        out = ROOT / "raw" / "active" / "ohlcv" / f"year={year}.parquet"
        df_all.to_parquet(out, index=False)
        print(f"✓ {year}년 저장: {len(df_all)}행, {df_all['code'].nunique()}종목")
    return {"year": year, "rows": len(df_all) if rows else 0, "fail": len(fail)}

print("함수 준비. 실행: collect_active_ohlcv_raw_by_year(2017), (2018), ...")

---
## Step D — Active Flow (배치, Step B와 동일 구조)

**매뉴얼 §5.1:** `get_market_trading_value_by_date(start, end, code)` 종목별 1콜.
**단위:** **원(KRW)** — 매뉴얼 §5.1 수량 표기는 오기. Notebook 1에서 확인.
**제로섬:** foreign+inst+indiv+etc ≈ 0.

In [ ]:
def run_active_flow_batch(batch_num):
    start = batch_num * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(codes))
    batch_codes = codes[start:end]
    print(f"=== Flow Batch {batch_num}: codes[{start}:{end}] ===")
    
    rows = []
    fail = []
    for i, code in enumerate(batch_codes, 1):
        try:
            df = krx.get_market_trading_value_by_date(START, END, code)
            if len(df) > 0:
                df = df.reset_index().rename(columns={"날짜": "date"})
                df["code"] = code.zfill(6)
                rows.append(df)
        except Exception as e:
            fail.append((code, str(e)[:40]))
        time.sleep(0.15)
        if i % 10 == 0:
            print(f"  진행: {i}/{len(batch_codes)}")
    
    if rows:
        df_all = pd.concat(rows, ignore_index=True)
        df_all["date"] = pd.to_datetime(df_all["date"])
        df_all["year"] = df_all["date"].dt.year
        for year, df_y in df_all.groupby("year"):
            out = ROOT / "raw" / "active" / "flow" / f"year={year}.parquet"
            df_y_new = df_y.drop(columns=["year"])
            if out.exists():
                df_old = pd.read_parquet(out)
                df_y_new = pd.concat([df_old, df_y_new], ignore_index=True).drop_duplicates(
                    subset=["code", "date"], keep="last"
                )
            df_y_new.to_parquet(out, index=False)
        print(f"✓ Flow Batch {batch_num}: {len(df_all)}행, {df_all['code'].nunique()}종목, 실패 {len(fail)}")
    return {"batch": batch_num, "rows": len(df_all) if rows else 0, "fail": len(fail)}

print("함수 준비. 실행: run_active_flow_batch(0), (1), ...")

---
## Step E — Active Foreign 지분율

**매뉴얼 §5.2:** `get_exhaustion_rates_of_foreign_investor(start, end, code)`.
**산출 컬럼:** foreign_hold_pct (외국인 지분율 %).

In [ ]:
def run_active_foreign_batch(batch_num):
    start = batch_num * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(codes))
    batch_codes = codes[start:end]
    print(f"=== Foreign Batch {batch_num}: codes[{start}:{end}] ===")
    
    rows = []
    fail = []
    for i, code in enumerate(batch_codes, 1):
        try:
            df = krx.get_exhaustion_rates_of_foreign_investor(START, END, code)
            if len(df) > 0:
                df = df.reset_index().rename(columns={"날짜": "date"})
                df["code"] = code.zfill(6)
                rows.append(df)
        except Exception as e:
            fail.append((code, str(e)[:40]))
        time.sleep(0.15)
        if i % 10 == 0:
            print(f"  진행: {i}/{len(batch_codes)}")
    
    if rows:
        df_all = pd.concat(rows, ignore_index=True)
        df_all["date"] = pd.to_datetime(df_all["date"])
        df_all["year"] = df_all["date"].dt.year
        for year, df_y in df_all.groupby("year"):
            out = ROOT / "raw" / "active" / "foreign" / f"year={year}.parquet"
            df_y_new = df_y.drop(columns=["year"])
            if out.exists():
                df_old = pd.read_parquet(out)
                df_y_new = pd.concat([df_old, df_y_new], ignore_index=True).drop_duplicates(
                    subset=["code", "date"], keep="last"
                )
            df_y_new.to_parquet(out, index=False)
        print(f"✓ Foreign Batch {batch_num}: {len(df_all)}행, {df_all['code'].nunique()}종목, 실패 {len(fail)}")
    return {"batch": batch_num, "rows": len(df_all) if rows else 0, "fail": len(fail)}

print("함수 준비. 실행: run_active_foreign_batch(0), (1), ...")

---
## Step F — Macro 수집

**확정:**
- KOSPI200: pykrx `1028` (매뉴얼 §6.1)
- VKOSPI: 이미 보유 (`data/raw/vkospi_10y.parquet`)
- KOSPI 종합 / KOSDAQ 종합 / KOSDAQ150: 검증 셀에서 확정한 코드 사용
- USD/KRW: FDR (매뉴얼 §6.3)
- 한국 기준금리: ECOS 722Y001 (매뉴얼 §6.4)

**저장:** `data/raw/macro/macro_10y.json` (utf-8-sig)

In [ ]:
# F-1. KOSPI200 (pykrx)
macro = {}

df_k200 = krx.get_index_ohlcv(START, END, "1028")
df_k200 = df_k200.reset_index().rename(columns={"날짜": "date", "종가": "close"})
macro["KOSPI200"] = {d.strftime("%Y-%m-%d"): float(c) for d, c in zip(df_k200["date"], df_k200["close"])}
print(f"KOSPI200: {len(macro['KOSPI200'])}일")

In [ ]:
# F-2. KOSPI 종합, KOSDAQ 종합, KOSDAQ150 — 검증 셀 결과 사용
# REPORT["steps"]["index_codes"]에서 코드 가져옴
index_codes = REPORT["steps"].get("index_codes", {})

for name, codes_found in index_codes.items():
    if not codes_found:
        continue
    # 가장 첫번째 매칭 사용
    ticker = codes_found[0]
    try:
        df = krx.get_index_ohlcv(START, END, ticker)
        df = df.reset_index().rename(columns={"날짜": "date", "종가": "close"})
        key = name.replace(" ", "").upper()  # "코스닥 150" → "코스닥150"
        macro[key] = {d.strftime("%Y-%m-%d"): float(c) for d, c in zip(df["date"], df["close"])}
        print(f"{name} ({ticker}): {len(macro[key])}일")
    except Exception as e:
        print(f"{name} 실패: {e}")
    time.sleep(0.3)

In [ ]:
# F-3. VKOSPI 합치기 (보유 데이터에서)
df_vk = pd.read_parquet(ROOT / "raw" / "vkospi_10y.parquet")
df_vk["date"] = pd.to_datetime(df_vk["date"])
macro["VKOSPI"] = {d.strftime("%Y-%m-%d"): float(c) for d, c in zip(df_vk["date"], df_vk["close"])}
print(f"VKOSPI: {len(macro['VKOSPI'])}일")

In [ ]:
# F-4. USD/KRW (FDR, 매뉴얼 §6.3)
try:
    fx = fdr.DataReader("USD/KRW", START[:4], END[:4])
    macro["USDKRW"] = {d.strftime("%Y-%m-%d"): float(c) for d, c in zip(fx.index, fx["Close"])}
    print(f"USD/KRW: {len(macro['USDKRW'])}일 (FDR)")
except Exception as e:
    print(f"USD/KRW FDR 실패: {e}")
    print("  → ECOS 우회 필요 (수동 추가)")

In [ ]:
# F-5. 한국 기준금리 (ECOS 722Y001, 매뉴얼 §6.4)
if ECOS_KEY:
    # ECOS API spec: /StatisticSearch/{key}/json/kr/start/limit/code/freq/start_date/end_date
    # 722Y001: 한국은행 기준금리, 주기 D (일별)
    url = f"https://ecos.bok.or.kr/api/StatisticSearch/{ECOS_KEY}/json/kr/1/10000/722Y001/D/{START[:4]}0101/{END[:4]}1231"
    try:
        resp = requests.get(url, timeout=30)
        data = resp.json()
        rows_ecos = data.get("StatisticSearch", {}).get("row", [])
        print(f"ECOS 응답: {len(rows_ecos)}행")
        if rows_ecos:
            macro["BASE_RATE"] = {
                f"{r['TIME'][:4]}-{r['TIME'][4:6]}-{r['TIME'][6:8]}": float(r["DATA_VALUE"])
                for r in rows_ecos
            }
            print(f"BASE_RATE: {len(macro['BASE_RATE'])}일")
        else:
            print("  → 응답 데이터 없음. ECOS 통계표 코드/주기 검증 필요.")
            print(f"  응답 일부: {str(data)[:300]}")
    except Exception as e:
        print(f"ECOS 실패: {e}")
else:
    print("ECOS_KEY 없음 — BASE_RATE 생략")

In [ ]:
# F-6. macro 저장
macro_path = ROOT / "raw" / "macro" / "macro_10y.json"
with open(macro_path, "w", encoding="utf-8-sig") as f:
    json.dump(macro, f, ensure_ascii=False, indent=2, default=str)
print(f"\n✓ Macro 저장: {macro_path}")
for k, v in macro.items():
    print(f"  {k}: {len(v)}일")

REPORT["steps"]["macro"] = {k: len(v) for k, v in macro.items()}

---
## Step G — 검증 리포트 + 최종 push

In [ ]:
# 검증
import glob

def summarize(label, glob_pattern):
    files = sorted(glob.glob(glob_pattern))
    if not files:
        return {"label": label, "files": 0}
    dfs = [pd.read_parquet(f) for f in files]
    df = pd.concat(dfs, ignore_index=True)
    return {
        "label": label,
        "files": len(files),
        "rows": int(len(df)),
        "codes": int(df["code"].nunique()) if "code" in df.columns else None,
        "date_min": str(df["date"].min())[:10] if "date" in df.columns else None,
        "date_max": str(df["date"].max())[:10] if "date" in df.columns else None,
    }

summary = {
    "active_ohlcv_adj": summarize("OHLCV adj", str(ROOT / "raw" / "active" / "ohlcv_adj" / "*.parquet")),
    "active_ohlcv_raw": summarize("OHLCV raw", str(ROOT / "raw" / "active" / "ohlcv" / "*.parquet")),
    "active_flow":      summarize("Flow",      str(ROOT / "raw" / "active" / "flow" / "*.parquet")),
    "active_foreign":   summarize("Foreign",   str(ROOT / "raw" / "active" / "foreign" / "*.parquet")),
}
REPORT["summary"] = summary

report_path = ROOT / "checks" / "notebook2_report.json"
with open(report_path, "w", encoding="utf-8-sig") as f:
    json.dump(REPORT, f, ensure_ascii=False, indent=2, default=str)
print(f"리포트 저장: {report_path}")
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# Final push (단계별로 매번 push 권장 — 이 셀은 일괄 push 예시)
import subprocess

origin_url = subprocess.run(["git", "remote", "get-url", "origin"], capture_output=True, text=True).stdout.strip()
auth_url = origin_url.replace("https://", f"https://{GH_TOKEN}@")

!git config user.email "msm@stanleyim.local"
!git config user.name "stanleyim"
!git remote set-url origin {auth_url}
!git add data/ msm_data_ingestion.ipynb
!git commit -m "Notebook 2 ingestion: universe + active OHLCV/Flow/Foreign + macro"
!git push origin main
!git remote set-url origin {origin_url}
print("\n✓ Notebook 2 push 완료")

---
## 다음 단계: Notebook 3 — State Vector Build

- delisted (285종목, 보유) + active (Universe 산출) 통합
- 5축 계산 (r, v, f, σ, ℓ)
- Dual normalization (cross-sectional + time-series)
- `data/state/state_vector.parquet`